# Barranquilla MOPAC: diagnóstico del QM/MM con MOPAC en la GPU

En la corrida anterior (`qmmm_gpu_threshold_a100`), MOPAC en la GPU tardó ~0,13 s por paso con regiones QM de 10 a
107 átomos: el tiempo no cambia con el tamaño, y los kernels registrados (hcore 2–4 ms, gradiente 6–100 ms) no lo
explican. Además, las pruebas NVE (0,2–0,4 ps) fueron demasiado cortas para comparar la conservación de la energía
de las rutas GPU y CPU.

| Bloque | Qué mide | Tiempo (A100) |
|---|---|---|
| A. Variantes, 10 átomos QM | chignolina residuo 1 en agua: el mismo cálculo con una cosa cambiada cada vez (sin salida de depuración, OpenMP con 1 hilo o en espera pasiva, PME en la CPU, hcore y gradiente en la CPU, SCF en la CPU) y con el perfil por secciones de MOPAC | ~15 min |
| B. Perfil, tripsina–BPTI | sitio activo (59 átomos QM): perfil por secciones en GPU y CPU | ~8 min |
| C. NVE de 2 ps | tripsina–BPTI, 0,5 fs (4000 pasos) y 1 fs (2000 pasos), MOPAC en GPU y en CPU; se guarda la energía total de cada paso | ~45 min |

En todas las corridas, un envoltorio de Python mide por llamada el tiempo de la fuerza QM/MM, de MOPAC
(`QMSession.step`), de la parte PME y el resto del paso (OpenMM). El código de MOPAC y de `run_qmmm_md.py` es el de la
corrida anterior (mismo commit).

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU A100*, y luego *Ejecutar todo*. Los
resultados se guardan en Google Drive (`MyDrive/mopac_paper_bench/RESULTS_NAME`); si la sesión se corta, ejecute de
nuevo todas las celdas y las corridas hechas se leen sin repetirse. La última celda descarga un zip (se puede
ejecutar en cualquier momento con lo que haya).

## 1. Configuración

In [ ]:
COMMIT = 'fb16b813'            # el mismo de qmmm_gpu_threshold_a100
RESULTS_NAME = 'qmmm_gpu_floor_a100'
USE_DRIVE = True
SEED = 11
DIAG_STEPS = 300               # bloques A y B
NVE_05, NVE_1 = 4000, 2000     # pasos NVE del bloque C (2 ps con 0,5 fs y con 1 fs)
RUN_NVE_1FS = True             # False: solo 0,5 fs en el bloque C (~20 min menos)
QUICK = False                  # True: 20 pasos (prueba del notebook)

## 2. GPU, CPU y carpeta de resultados

In [ ]:
import os, re, subprocess, sys, time, shutil, json, gzip
from pathlib import Path

def run(cmd, check=True, env=None, log=None):
    # Run a shell command and stream its output to the cell (and to a log file)
    print('$', cmd, flush=True)
    full = dict(os.environ); full.update(env or {})
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash', stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, env=full)
    fh = open(log, 'w') if log else None
    for line in p.stdout:
        print(line, end='', flush=True)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if check and p.returncode != 0:
        raise RuntimeError(f'exit {p.returncode}: {cmd}')
    return p.returncode

run('nvidia-smi')
gpu = subprocess.run('nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader', shell=True,
                     capture_output=True, text=True).stdout.strip()
cpu = subprocess.run("lscpu | grep 'Model name' | sed 's/.*: *//'", shell=True, capture_output=True,
                     text=True).stdout.strip()
NCPU = os.cpu_count()
print(f'GPU: {gpu}\nCPU: {cpu}, {NCPU} núcleos lógicos')
CC = gpu.split(',')[-1].strip().replace('.', '')

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = Path('/content/drive/MyDrive/mopac_paper_bench') / RESULTS_NAME
else:
    OUT = Path('/content/mopac_paper_bench') / RESULTS_NAME
OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'env.json').write_text(json.dumps({'gpu': gpu, 'cpu': cpu, 'ncpu': NCPU, 'commit': COMMIT}, indent=1))
print('resultados en', OUT)

## 3. Compilar MOPAC (biblioteca) con OpenBLAS

~10 minutos.

In [ ]:
SRC, BUILD = Path('/content/mopac_gpu'), Path('/content/build')
run('apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq gfortran ninja-build '
    'libopenblas-dev liblapack-dev > /dev/null')
os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ['PATH']
shutil.rmtree(SRC, ignore_errors=True); shutil.rmtree(BUILD, ignore_errors=True)
run(f'git clone -q https://github.com/juvenalyosa/mopac_gpu.git {SRC}')
run(f'git -C {SRC} checkout -q {COMMIT}')
run(f'git -C {SRC} log -1 --format="%h %ad %s" --date=short')
head = subprocess.run(f'git -C {SRC} rev-parse HEAD', shell=True, capture_output=True, text=True).stdout.strip()
if not head.startswith(COMMIT):
    raise SystemExit(f'HEAD {head} no es COMMIT {COMMIT}')
stamp = OUT / 'commit.txt'
if stamp.exists() and not head.startswith(stamp.read_text().strip()[:8]):
    raise SystemExit(f'{OUT} tiene resultados de {stamp.read_text().strip()}; use otro RESULTS_NAME')
stamp.write_text(head + '\n')
t0 = time.time()
run(f'cmake -S {SRC} -B {BUILD} -G Ninja -DGPU=ON -DTESTS=OFF -DCMAKE_BUILD_TYPE=Release -DCUDA_ARCHS={CC} '
    f'-DENABLE_MKL=OFF -DBLA_VENDOR=OpenBLAS > /content/cmake.log 2>&1 || (tail -40 /content/cmake.log; exit 1)')
run(f'cmake --build {BUILD} --target mopac --parallel {NCPU} > /content/build.log 2>&1 '
    f'|| (grep -i -B2 -A6 error /content/build.log | head -80; exit 1)')
LIBMOPAC = next(BUILD.glob('libmopac.so*'))
print(f'compilado en {time.time() - t0:.0f} s: {LIBMOPAC}')

## 4. OpenMM y CuPy

In [ ]:
m = re.search(r'release (\d+)', subprocess.run('nvcc --version', shell=True, capture_output=True, text=True).stdout)
cupy_pkg = f'cupy-cuda{m.group(1) if m else 12}x'
other = [p for p in subprocess.run('pip list 2>/dev/null', shell=True, capture_output=True, text=True).stdout.split()
         if p.startswith('cupy-cuda') and p != cupy_pkg]
if other:
    run('pip uninstall -y -q ' + ' '.join(other), check=False)
run(f"pip install -q 'openmm[cuda12]==8.6.1' {cupy_pkg} 2>&1 | tail -2", check=False)
run(f"{sys.executable} -c \"import openmm as m; print('OpenMM', m.__version__, "
    f"[m.Platform.getPlatform(i).getName() for i in range(m.Platform.getNumPlatforms())])\"", check=False)
run(f"cd {SRC}/scripts/qmmm && {sys.executable} -c \"from qmmm_pme import gpu_available; "
    f"print('QM-PME grid on the GPU (CuPy):', gpu_available())\"", check=False)

## 5. Lanzador con tiempos por llamada

`/content/qmmm_prof.py` ejecuta `run_qmmm_md.py` sin cambiarlo y envuelve cuatro métodos para medir cada llamada:
`MopacQMMMForce.compute` (toda la fuerza QM/MM), `QMSession.step` (MOPAC), `_pme_far_potential` y
`_pme_far_gradient` (PME). El tiempo entre el final de una llamada a la fuerza y el comienzo de la siguiente es el
resto del paso (OpenMM). Los tiempos se guardan en `pytimes.json`.

Modos: `cpu` = `MOPAC_NOGPU=1`; `gpu` = `MOPAC_FORCEGPU=1`. Cada variante añade variables de entorno o opciones.
Con `MOPAC_GPU_PROFILE=1`, MOPAC escribe en su salida (`MOPAC_API_OUTPUT`) el tiempo acumulado por sección después
de cada paso; se guarda el último valor de cada sección en `sections.csv` y la salida completa comprimida.

In [ ]:
PROF = Path('/content/qmmm_prof.py')
PROF.write_text('''
import json, os, runpy, sys, time
script = sys.argv[1]
sys.argv = sys.argv[1:]
sys.path.insert(0, os.path.dirname(os.path.abspath(script)))
import openmm_mopac as om
import mopac_qmmm_api as api
T = {"t0": [], "compute": [], "mopac": [], "pme_far": [], "pme_grad": []}
def wrap(cls, name, key, stamp=False):
    f = getattr(cls, name)
    def g(self, *a, **k):
        t = time.perf_counter()
        if stamp:
            T["t0"].append(t)
        try:
            return f(self, *a, **k)
        finally:
            T[key].append(time.perf_counter() - t)
    setattr(cls, name, g)
wrap(om.MopacQMMMForce, "compute", "compute", True)
wrap(om.MopacQMMMForce, "_pme_far_potential", "pme_far")
wrap(om.MopacQMMMForce, "_pme_far_gradient", "pme_grad")
wrap(api.QMSession, "step", "mopac")
try:
    runpy.run_path(script, run_name="__main__")
finally:
    out = os.environ.get("QMMM_PROF_OUT")
    if out:
        with open(out, "w") as fh:
            json.dump(T, fh)
''')

RX = {
    'system': re.compile(r'system: (\d+) atoms, (\d+) residues, box ([0-9.]+) A'),
    'qm': re.compile(r'QM region: (\d+) atoms \+ (\d+) link atoms, charge (-?\d+); long-range QM-MM electrostatics: (.+)'),
    'nvt': re.compile(r'QM/MM NVT: (\d+) steps x ([0-9.]+) fs in ([0-9.]+) s = ([0-9.]+) s/step, ([0-9.]+) ps/day'),
    'nve': re.compile(r'NVE check (\d+) steps: drift (-?[0-9.]+) kJ/mol, slope of a linear fit (-?[0-9.]+) kJ/mol/ps, '
                      r'largest step ([0-9.]+), rms step ([0-9.]+), rms deviation from the fit ([0-9.]+)'),
    'sec': re.compile(r'\[PROFILE\] MOZYME_SECTION name=(\S+) calls=(\d+) ms=([0-9.]+)'),
}
QMMM = SRC / 'scripts' / 'qmmm' / 'run_qmmm_md.py'

def py_breakdown(T, first, last):
    # medians (ms) of the per-call times over calls first..last-1 (the NVT steps)
    import numpy as np
    c_all, t0 = np.array(T['compute']), np.array(T['t0'])
    c = c_all[first:last]
    # rest of the step: from the end of one force call to the start of the next
    j = np.arange(first, min(last, len(t0) - 1))
    gap = t0[j + 1] - (t0[j] + c_all[j])
    r = {'py_compute_ms': np.median(c) * 1e3 if len(c) else None,
         'py_compute_mean_ms': np.mean(c) * 1e3 if len(c) else None,
         'py_mopac_ms': np.median(T['mopac'][first:last]) * 1e3 if T['mopac'][first:last] else None,
         'py_mopac_mean_ms': np.mean(T['mopac'][first:last]) * 1e3 if T['mopac'][first:last] else None,
         'py_pme_ms': (np.median(np.array(T['pme_far'][first:last]) + np.array(T['pme_grad'][first:last])) * 1e3
                       if T['pme_far'][first:last] else 0.0),
         'py_rest_ms': np.median(gap) * 1e3 if len(gap) else None}
    return {k: (round(float(v), 2) if v is not None else None) for k, v in r.items()}

def qmmm(exp, case, variant, pdb, opts, steps, nve=0, mm_ps=10.0, env_extra=None):
    mode = variant.split('_')[0]
    env_extra = dict(env_extra or {})
    d = OUT / 'runs' / exp / case / variant
    cmd = (f'{sys.executable} -u {PROF} {QMMM} {pdb} --libmopac {LIBMOPAC} {opts} --platform CUDA '
           f'--steps {steps} --nve {nve} --mm-equil-ps {mm_ps} --seed {SEED}')
    key = cmd + '\n' + json.dumps(env_extra, sort_keys=True)
    if (d / 'result.json').exists() and (d / 'cmd.txt').exists() and (d / 'cmd.txt').read_text() == key:
        r = json.loads((d / 'result.json').read_text())
        print(f'[{exp}/{case}] {variant}: ya hecho, {r.get("s_per_step")} s/step')
        return r
    shutil.rmtree(d, ignore_errors=True); d.mkdir(parents=True)
    (d / 'cmd.txt').write_text(key)
    api_out = Path('/tmp') / f'mopac_api_{exp}_{case}_{variant}.out'
    api_out.unlink(missing_ok=True)
    env = {k: v for k, v in os.environ.items() if k not in ('MOPAC_NOGPU', 'MOPAC_FORCEGPU')}
    env.update(OMP_NUM_THREADS=str(NCPU), OPENBLAS_NUM_THREADS=str(NCPU), MOPAC_GPU_DEBUG='1',
               MOPAC_API_OUTPUT=str(api_out), QMMM_PROF_OUT=str(d / 'pytimes.json'))
    env['MOPAC_NOGPU' if mode == 'cpu' else 'MOPAC_FORCEGPU'] = '1'
    env.update(env_extra)
    env = {k: v for k, v in env.items() if v is not None}       # None: remove the variable
    print(f'[{exp}/{case}] {variant} {env_extra}', flush=True)
    t0 = time.perf_counter()
    with open(d / 'log.txt', 'w') as fh:
        rc = subprocess.run(cmd, shell=True, cwd=d, env=env, stdout=fh, stderr=subprocess.STDOUT).returncode
    text = (d / 'log.txt').read_text(errors='ignore')
    r = {'exp': exp, 'case': case, 'variant': variant, 'mode': mode, 'env': json.dumps(env_extra), 'rc': rc,
         'total_s': round(time.perf_counter() - t0, 1)}
    if (m := RX['system'].search(text)):
        r.update(total_atoms=int(m.group(1)), box_A=float(m.group(3)))
    if (m := RX['qm'].search(text)):
        r.update(qm_atoms=int(m.group(1)), link_atoms=int(m.group(2)), qm_charge=int(m.group(3)), long_range=m.group(4).strip())
    if (m := RX['nvt'].search(text)):
        r.update(steps=int(m.group(1)), dt_fs=float(m.group(2)), md_s=float(m.group(3)),
                 s_per_step=float(m.group(4)), ps_per_day=float(m.group(5)))
    if (m := RX['nve'].search(text)):
        r.update(nve_steps=int(m.group(1)), nve_drift_kj=float(m.group(2)), nve_slope_kj_ps=float(m.group(3)),
                 nve_rms_dev_kj=float(m.group(6)))
    if (d / 'pytimes.json').exists():
        T = json.loads((d / 'pytimes.json').read_text())
        r['force_calls'] = len(T['compute'])
        r.update(py_breakdown(T, 2, 2 + steps))          # the first 2 calls: first SCF from the Lewis structure
    lines = api_out.read_text(errors='ignore').splitlines() if api_out.exists() else []
    r['mozyme_gpu_lines'] = sum('[MOZYME GPU' in l for l in lines)
    r['gpu_scf_stdout_lines'] = sum('[MOZYME GPU SCF]' in l for l in text.splitlines())
    sec = {}
    for l in lines:
        if (m := RX['sec'].search(l)):
            sec[m.group(1)] = (int(m.group(2)), float(m.group(3)))
    if sec:
        with open(d / 'sections.csv', 'w') as fh:
            fh.write('section,calls,ms_total\n')
            for k, (c, ms) in sorted(sec.items(), key=lambda x: -x[1][1]):
                fh.write(f'{k},{c},{ms:.3f}\n')
    if api_out.exists():
        with open(api_out, 'rb') as fi, gzip.open(d / 'mopac_api.out.gz', 'wb') as fo:
            shutil.copyfileobj(fi, fo)
        api_out.unlink()
    (d / ('result.json' if rc == 0 and 's_per_step' in r else 'result_failed.json')).write_text(json.dumps(r, indent=1))
    print(f'    {r.get("s_per_step")} s/step; per call (ms, median): force {r.get("py_compute_ms")}, MOPAC '
          f'{r.get("py_mopac_ms")}, PME {r.get("py_pme_ms")}, rest {r.get("py_rest_ms")}; [MOZYME GPU] lines '
          f'{r["mozyme_gpu_lines"]}, rc {rc}, {r["total_s"]:.0f} s', flush=True)
    if rc != 0:
        print('\n'.join(text.splitlines()[-10:]))
    return r

def get_pdb(code):
    import urllib.request
    f = OUT / 'runs' / '_pdb' / f'{code}.pdb'
    f.parent.mkdir(parents=True, exist_ok=True)
    if not f.exists():
        urllib.request.urlretrieve(f'https://files.rcsb.org/download/{code}.pdb', f)
    return f

CHIG = SRC / 'tests' / 'data' / 'mozyme_gpu_reference' / '1UAO_opt.pdb'
ENZ = ('--qm-residues E:57:sc,E:102:sc,E:195:sc,I:15,I:16 --variants E:57=HID --padding 1.0 --ron 10 --roff 12 '
       '--distances E:195:OG-I:15:C,E:57:NE2-E:195:OG,E:57:HD1-E:102:OD2')
ENZ_PDB = get_pdb('2PTC')
NSTEP = 20 if QUICK else DIAG_STEPS

## A. Variantes con 10 átomos QM (chignolina, residuo 1)

La región más pequeña del barrido anterior: CPU 0,021 s/paso, GPU 0,132 s/paso. Cada variante cambia una sola cosa
respecto de `gpu` (la corrida anterior):

| Variante | Cambio | Pregunta |
|---|---|---|
| `cpu`, `gpu` | ninguno | ¿se repiten 0,021 y 0,132 s/paso en esta sesión? |
| `gpu_nodebug` | sin `MOPAC_GPU_DEBUG` | ¿cuesta la salida de depuración? |
| `gpu_omp1` | `OMP_NUM_THREADS=1` | ¿compiten los hilos de OpenMP con la GPU? |
| `gpu_passive` | `OMP_WAIT_POLICY=PASSIVE` | ¿hilos de OpenMP en espera activa? |
| `gpu_pmecpu` | `--pme-cpu` | ¿interfiere CuPy (PME) con MOPAC en la GPU? |
| `gpu_hcgradcpu` | hcore y gradiente en la CPU | ¿está el piso en esos dos pasos? |
| `gpu_scfcpu` | SCF en la CPU (hcore y gradiente en la GPU) | ¿está el piso en el SCF? |
| `gpu_prof`, `cpu_prof` | `MOPAC_GPU_PROFILE=1` | tiempo por sección dentro de MOPAC |

Las variables de `gpu_scfcpu` apagan las rutas del SCF en la GPU que activa la sesión QM/MM
(`mozyme_gpu_apply_default_environment` solo pone las que no están definidas); `sections.csv` y el número de
líneas `[MOZYME GPU SCF]` muestran qué corrió en realidad.

In [ ]:
SCF_OFF = {k: '0' for k in ('MOZYME_GPU_FORCE', 'MOPAC_MOZYME_SCF_EXPERIMENTAL', 'MOPAC_MOZYME_RESIDENT_SCF',
                            'MOPAC_MOZYME_SCF_GPU', 'MOPAC_MOZYME_RESIDENT_FOCK_GPU', 'MOPAC_MOZYME_MAKVEC_GPU')}
VARIANTS = [
    ('cpu', {}, ''), ('gpu', {}, ''),
    ('gpu_nodebug', {'MOPAC_GPU_DEBUG': None}, ''),
    ('gpu_omp1', {'OMP_NUM_THREADS': '1', 'OPENBLAS_NUM_THREADS': '1'}, ''),
    ('gpu_passive', {'OMP_WAIT_POLICY': 'PASSIVE'}, ''),
    ('gpu_pmecpu', {}, ' --pme-cpu'),
    ('gpu_hcgradcpu', {'MOPAC_MOZYME_HCORE_GPU': '0', 'MOPAC_MOZYME_GRAD_GPU': '0'}, ''),
    ('gpu_scfcpu', SCF_OFF, ''),
    ('gpu_prof', {'MOPAC_GPU_PROFILE': '1'}, ''),
    ('cpu_prof', {'MOPAC_GPU_PROFILE': '1'}, ''),
]
diag = []
for variant, envx, extra in VARIANTS:
    diag.append(qmmm('A_small', 'chig_1-1', variant, CHIG, '--qm-residues 1 --padding 2.5 --ron 6 --roff 8' + extra,
                     NSTEP, env_extra=envx))

## B. Perfil de la tripsina–BPTI (59 átomos QM)

Las opciones de la corrida del artículo, 300 pasos de 0,5 fs, con `MOPAC_GPU_PROFILE=1` en GPU y en CPU, y la GPU con
OpenMP en espera pasiva.

In [ ]:
for variant, envx in (('gpu_prof', {'MOPAC_GPU_PROFILE': '1'}), ('cpu_prof', {'MOPAC_GPU_PROFILE': '1'}),
                      ('gpu_passive', {'OMP_WAIT_POLICY': 'PASSIVE'})):
    diag.append(qmmm('B_trypsin', 'trypsin_pme', variant, ENZ_PDB, ENZ, NSTEP,
                     mm_ps=10.0 if QUICK else 100.0, env_extra=envx))

## C. NVE de 2 ps, tripsina–BPTI, MOPAC en la GPU y en la CPU

400 pasos NVT y luego NVE (Verlet) de 4000 pasos con 0,5 fs y de 2000 pasos con 1 fs. `nve_energy.txt` guarda la
energía total de cada paso (kJ/mol).

In [ ]:
nve_cases = [('dt05', '', NVE_05)] + ([('dt1', ' --dt 1.0', NVE_1)] if RUN_NVE_1FS else [])
for case, extra, nve in nve_cases:
    for variant in ('gpu', 'cpu'):
        diag.append(qmmm('C_nve', f'trypsin_{case}', variant, ENZ_PDB, ENZ + extra, 20 if QUICK else 400,
                         nve=40 if QUICK else nve, mm_ps=10.0 if QUICK else 100.0))

## 6. Resultados

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
rows = []
for f in sorted((OUT / 'runs').rglob('result*.json')):
    r = json.loads(f.read_text()); r['ok'] = f.name == 'result.json'; rows.append(r)
df = pd.DataFrame(rows)
df.to_csv(OUT / 'qmmm_floor.csv', index=False)
cols = ['exp', 'case', 'variant', 's_per_step', 'py_compute_ms', 'py_mopac_ms', 'py_mopac_mean_ms', 'py_pme_ms',
        'py_rest_ms', 'mozyme_gpu_lines', 'gpu_scf_stdout_lines', 'nve_steps', 'nve_drift_kj', 'nve_slope_kj_ps',
        'nve_rms_dev_kj', 'ok']
display(df[[c for c in cols if c in df]])

# sections inside MOPAC (profile runs), ms per force call
for f in sorted((OUT / 'runs').rglob('sections.csv')):
    r = json.loads((f.parent / 'result.json').read_text()) if (f.parent / 'result.json').exists() else {}
    s = pd.read_csv(f)
    s['ms_per_call'] = s['ms_total'] / max(1, r.get('force_calls', 1))
    print(f'\n{f.parent.relative_to(OUT / "runs")}: {r.get("force_calls")} force calls')
    display(s.head(25))

# NVE: total energy, CPU and GPU
nf = sorted((OUT / 'runs' / 'C_nve').rglob('nve_energy.txt'))
if nf:
    fig, ax = plt.subplots(figsize=(6, 3.4))
    for f in nf:
        e = np.loadtxt(f); case, variant = f.parent.parent.name, f.parent.name
        dt = 1.0 if 'dt1' in case else 0.5
        t = np.arange(len(e)) * dt / 1000
        ax.plot(t, e - e[0], lw=0.8, label=f'{case} {variant}')
    ax.set(xlabel='t (ps)', ylabel='E total − E(0) (kJ/mol)')
    ax.legend(fontsize=7, frameon=False)
    fig.tight_layout(); fig.savefig(OUT / 'qmmm_nve.png', dpi=200); plt.show()

## 7. Descargar

In [ ]:
tmp = Path('/content/_qmmm_floor') / RESULTS_NAME
shutil.rmtree(tmp.parent, ignore_errors=True)
tmp.mkdir(parents=True)
for f in ('qmmm_floor.csv', 'qmmm_nve.png', 'env.json', 'commit.txt'):
    if (OUT / f).exists():
        shutil.copy2(OUT / f, tmp / f)
for pat in ('*.json', 'log.txt', 'cmd.txt', 'sections.csv', 'nve_energy.txt', 'mopac_api.out.gz'):
    for f in (OUT / 'runs').rglob(pat):
        dst = tmp / 'runs' / f.relative_to(OUT / 'runs')
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, dst)
z = shutil.make_archive(f'/content/{RESULTS_NAME}', 'zip', tmp.parent)
print(z, f'{os.path.getsize(z) / 1e6:.1f} MB')
from google.colab import files
files.download(z)